# LanceDB fundamentals

LanceDB is an open-source vector database designed to handle large-scale vector data efficiently. It provides a robust platform for storing, indexing, and querying high-dimensional vectors, making it ideal for applications in machine learning, computer vision, and natural language processing.

We'll be using LanceDB for the purpose of creating RAG applications. However before we dive into that, let's cover some fundamental concepts of LanceDB, so that you know what is happening under the hood.

## Connect LanceDB

- when connecting to LanceDB it will create a folder for your lancedb database if it does not already exist

In [1]:
import lancedb

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='/Users/aigineer/Documents/teaching repos/llmops_course/09_lancedb/vector_database')

In [2]:
db.uri

'/Users/aigineer/Documents/teaching repos/llmops_course/09_lancedb/vector_database'

## Create a Table

when creating a table, it is stored as a folder with .lance suffix containing the following folders
- _transactions
- _versions
- data

a table is a pyarraow table stored in the data folder

In [3]:
import json
with open("animals_text_embeddings.json") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
table_animals = db.create_table("animals_text", exist_ok=True, data=data)
table_animals


LanceTable(name='animals_text', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/teaching repos/llmops_course/09_lancedb/vector_database'))

In [5]:
# it's a pyarrow table 
table_animals.head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

In [6]:
# converting to pandas 
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


add some more data to the table

In [7]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=2)

In [8]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table and then delete it

In [9]:
# LanceModel is a Pydantic model that can be converted to LanceDB table
from lancedb.pydantic import LanceModel

class JokeSchema(LanceModel):
    joke: str 
    rating: int


db.create_table(name = "jokes",schema=JokeSchema, exist_ok=True)

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/teaching repos/llmops_course/09_lancedb/vector_database'))

In [10]:
db.table_names()

/var/folders/tj/m_tpmk9937l71skgpnzqylbm0000gn/T/ipykernel_55153/763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

In [11]:
db.drop_table("jokes")

In [12]:
db.table_names()

/var/folders/tj/m_tpmk9937l71skgpnzqylbm0000gn/T/ipykernel_55153/763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text']

## Open existing table

In [13]:
# since db is already connected to vector_database it has animals_text as table
db.open_table("animals_text").head()

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa.","A large gray elephant drinking water.","A fast cheetah sprinting across the savannah.","A colorful parrot perched on a branch."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1],[0.88,0.22,0.55],[0.95,0.12,0.72],[0.25,0.66,0.81]]]

## Vector search in LanceDB

Uses ANN - approximate nearest neighbours for searching vectors

- search with vector directly
- note that the vector column is already an example embedding simulated with LLM

A more realistic approach is to just have text records, use an existing embedding model and embedd those text records, then embed the query vector and find closest match 


In [14]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [15]:
query_vector = [0.5, 0.2, 0.9]

# finds closest rows based on the Euclidean distance by default
table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673


## Embeddings API

LanceDB provides an Embeddings API that allows you to easily generate and store vector embeddings for your data. This is particularly useful for applications such as semantic search, recommendation systems, and clustering.

We'll use embeddings model from sentence-transformer here

In [ ]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry


model = get_registry().get("cohere").create(name = "embed-multilingual-light-v3.0")


class JokeModel(LanceModel):
    joke: str = model.SourceField() # marks text as input to embedding function
    vector: Vector(model.ndims()) = model.VectorField() # vector is destination of computed embedding

table_jokes = db.create_table("jokes", schema = JokeModel, exist_ok=True)
table_jokes



/Users/aigineer/Documents/teaching repos/llmops_course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9523.60it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/aigineer/Documents/teaching repos/llmops_course/09_lancedb/vector_database'))

In [18]:
import pandas as pd 
with open("jokes.json", "r") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


adds data to our table

In [19]:


table_jokes.add(df_jokes)


AddResult(version=2)

we can see that vector column has been created automatically and embeddings have been generated for each joke

In [20]:
table_jokes.head(2)

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[1024]
  child 0, item: float
----
joke: [["Parallel lines have so much in common—it’s sad they’ll never meet.","ETL stands for “Extract, Transform, Leave for the next person.”"]]
vector: [[[0.013246294,0.016852118,-0.028178778,-0.06076251,0.043403603,...,0.012054106,0.0141222635,0.0033709456,-0.010680341,0.028041745],[0.02986249,-0.00056345825,-0.008892237,-0.057397082,0.04463922,...,-0.010029852,0.010198969,-0.015285406,-0.030706173,-0.005296866]]]

In [21]:
table_jokes.to_pandas().head()

,joke,vector
0,Parallel lines have so much in common—it’s sad...,"[0.013246294, 0.016852118, -0.028178778, -0.06..."
1,"ETL stands for “Extract, Transform, Leave for ...","[0.02986249, -0.00056345825, -0.008892237, -0...."
2,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0..."
3,"Gold walks into a bar. The bartender says, “Au...","[0.046327822, 0.0009904793, -0.041676145, -0.0..."
4,C# devs don’t argue; they just throw exceptions.,"[0.013506025, -0.006613272, -0.013990911, -0.0..."


In [22]:
table_jokes.search("data engineering jokes").limit(8).to_pandas()

,joke,vector,_distance
0,"Data engineer motto: If it works, don’t touch ...","[0.013415303, -0.030821968, -0.04201837, -0.02...",0.299088
1,Why do data engineers hate nature? Too many un...,"[0.001068325, -0.01199555, -0.034374196, -0.03...",0.302485
2,I asked the data lake if it had my file. It sa...,"[0.017539004, -0.017620457, -0.033351053, -0.0...",0.319631
3,I told a chemistry joke… there was no reaction.,"[0.009617329, -0.0058871307, -0.01950332, -0.0...",0.333617
4,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0...",0.338996
5,"Our data pipeline broke… but don't worry, it w...","[0.006907674, -0.022554241, -0.03454011, -0.03...",0.339771
6,Why did the Python programmer get bitten? Beca...,"[0.00027592052, 0.00048585594, -0.04955295, -0...",0.349114
7,The C# compiler walked into a bar. The bartend...,"[0.041754726, -0.0024566723, -0.040765837, -0....",0.349219


In [23]:
table_jokes.search("give me jokes on C#").limit(8).to_pandas()

,joke,vector,_distance
0,The C# compiler walked into a bar. The bartend...,"[0.041754726, -0.0024566723, -0.040765837, -0....",0.246672
1,Why is C# like a musical? It has so many classes.,"[0.008130425, 0.0036474017, -0.02696161, -0.02...",0.269825
2,C# devs don’t argue; they just throw exceptions.,"[0.013506025, -0.006613272, -0.013990911, -0.0...",0.270728
3,Why did the C# developer go broke? He kept usi...,"[-0.0032815428, -7.01899e-05, -0.027963601, -0...",0.271089
4,C# programmers love coffee—they're always work...,"[0.022888584, -0.0021838509, -0.02993935, -0.0...",0.312129
5,Why did the Python programmer get bitten? Beca...,"[0.00027592052, 0.00048585594, -0.04955295, -0...",0.321323
6,I told a chemistry joke… there was no reaction.,"[0.009617329, -0.0058871307, -0.01950332, -0.0...",0.322294
7,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0...",0.323483


In [24]:
table_jokes.search("give me chemistry related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[0.009617329, -0.0058871307, -0.01950332, -0.0...",0.269470
1,Why did the chemist ground his kids? Because t...,"[0.036111806, -0.002387372, -0.034442466, -0.0...",0.285143
2,Why’s 6 afraid of 7? Because 7 8 9.,"[0.039057843, 0.012996828, -0.016367558, -0.03...",0.312969
3,Oxygen and Magnesium got together—OMg!,"[0.042133663, -0.010085303, -0.036200028, -0.0...",0.315217
4,Chemists have all the solutions… mostly in bea...,"[0.03842467, -0.0011145632, -0.038784124, -0.0...",0.323920
5,"Gold walks into a bar. The bartender says, “Au...","[0.046327822, 0.0009904793, -0.041676145, -0.0...",0.324595
6,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0...",0.332331
7,Never trust an atom—they make up everything.,"[0.043455284, 0.007818969, -0.026873162, -0.03...",0.339600


In [25]:
table_jokes.search("give me nature related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[0.001068325, -0.01199555, -0.034374196, -0.03...",0.308321
1,Why did the Python programmer get bitten? Beca...,"[0.00027592052, 0.00048585594, -0.04955295, -0...",0.326960
2,I told a chemistry joke… there was no reaction.,"[0.009617329, -0.0058871307, -0.01950332, -0.0...",0.330260
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.039057843, 0.012996828, -0.016367558, -0.03...",0.332731
4,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0...",0.333455
5,"Gold walks into a bar. The bartender says, “Au...","[0.046327822, 0.0009904793, -0.041676145, -0.0...",0.349089
6,Why did the chemist ground his kids? Because t...,"[0.036111806, -0.002387372, -0.034442466, -0.0...",0.365028
7,The C# compiler walked into a bar. The bartend...,"[0.041754726, -0.0024566723, -0.040765837, -0....",0.372068


## Hybrid search

hybrid search combines traditional keyword-based search with vector-based search to provide more relevant results. This approach leverages the strengths of both methods, allowing for more accurate and context-aware retrieval of information.

In [26]:
# to enable keyword based search on joke column
table_jokes.create_fts_index("joke", replace=True)


In [27]:
from lancedb import rerankers

# reranks the results from a vector and full text search
reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "give me nature related jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="joke",
).rerank(reranker).limit(8).to_pandas()

# somewhat different to before
results


,joke,vector,_relevance_score
0,Why do data engineers hate nature? Too many un...,"[0.001068325, -0.01199555, -0.034374196, -0.03...",0.032522
1,I told a chemistry joke… there was no reaction.,"[0.009617329, -0.0058871307, -0.01950332, -0.0...",0.032266
2,Why did the Python programmer get bitten? Beca...,"[0.00027592052, 0.00048585594, -0.04955295, -0...",0.016129
3,Why’s 6 afraid of 7? Because 7 8 9.,"[0.039057843, 0.012996828, -0.016367558, -0.03...",0.015625
4,What do you call a snake that runs your script...,"[-0.0055703004, 0.0110165095, -0.052640483, -0...",0.015385
5,"Gold walks into a bar. The bartender says, “Au...","[0.046327822, 0.0009904793, -0.041676145, -0.0...",0.015152
6,Why did the chemist ground his kids? Because t...,"[0.036111806, -0.002387372, -0.034442466, -0.0...",0.014925
7,The C# compiler walked into a bar. The bartend...,"[0.041754726, -0.0024566723, -0.040765837, -0....",0.014706


## Rule of thumb 

- For exact matching -> FTS
- For meaning based matching -> Vector search
- Unpredictable or mixed queries -> Hybrid search
